In [2]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cna, glob, os
import vima

In [3]:
def test_clusters(df, cols, pheno, donor, Nnull=1000):
    X = df[cols].values.copy()
    pheno = df[pheno].copy()
    donorids = df[donor].copy()
    
    filter = np.isfinite(pheno)
    X = X[filter,:]
    pheno = pheno[filter]
    donorids = donorids[filter]
    
    X = (X - X.mean(axis=0))/X.std(axis=0)
    y_ = cna.tl._stats.grouplevel_permutation(donorids, pheno, Nnull).astype('float')
    pheno = (pheno - pheno.mean())/pheno.std()
    y_ -= y_.mean(axis=0)
    y_ /= y_.std(axis=0)
    
    ncorrs = np.nan_to_num(X.T.dot(pheno) / len(X))
    nullncorrs = np.nan_to_num(X.T.dot(y_) / len(X))
    pvals = ((np.abs(nullncorrs) >= np.abs(ncorrs)[:,None]).sum(axis=1) + 1)/(Nnull + 1)
    globalp = (((ncorrs**2).sum() <= (nullncorrs**2).sum(axis=0)).sum() + 1)/(Nnull + 1)
    
    maxcorr = max(np.abs(ncorrs).max(), 0.001)
    fdr_thresholds = np.arange(maxcorr/4, maxcorr, maxcorr/400)
    fdr_vals = cna.tl._stats.empirical_fdrs(ncorrs, nullncorrs, fdr_thresholds)

    fdrs = pd.DataFrame({
        'threshold':fdr_thresholds,
        'fdr':fdr_vals,
        'num_detected': [(np.abs(ncorrs)>t).sum() for t in fdr_thresholds]})
    if len(fdrs[fdrs.fdr <= 0.1]) > 0:
        fdr10pt = fdrs[fdrs.fdr <= 0.1].threshold.min()
    else:
        fdr10pt = np.infty
    return fdrs, fdr10pt, ncorrs, pvals, globalp

In [28]:
from scipy.stats import entropy
def integration(d):
    A = d.obsp['connectivities']
    A /= A.sum(axis=1)
    S = pd.get_dummies(d.obs.sid).astype(np.float32)
    baseline = np.power(2, entropy(d.obs.sid.value_counts() / len(d), base=2))
    perplexities = np.power(2, entropy(np.array(A.dot(S)), axis=1, base=2)) / baseline
    d.obs['perplexity'] = perplexities

def test_cluster_cc(d, samplemeta, secondary_pheno=None):
    # Determine cluster key
    if 'cluster_method' in d.obs.columns:
        cluster_key = 'cluster_method'
    elif 'leiden_1' in d.obs.columns:
        cluster_key = 'leiden_1'
    else:
        cluster_key = [c for c in d.obs.columns if c.startswith('leiden')][-1]
    print(f'Using {cluster_key} for clustering. There are {d.obs[cluster_key].nunique()} clusters')

    # Build crosstab and normalize
    ct = pd.crosstab(d.obs['sid'], d.obs[cluster_key]).div(
        pd.crosstab(d.obs['sid'], d.obs[cluster_key]).sum(axis=1), axis=0)
    ct.index.name = 'sid'
    clusts = ct.columns.values
    
    if secondary_pheno:
        ct['case2'] = samplemeta[secondary_pheno]

    # Helper to run test_clusters and store results
    def store_results(suffix, pheno, mask=None):
        cols = clusts if mask is None else clusts[mask]
        if len(cols) > 0:
            myct = ct[cols].div(ct[cols].sum(axis=1), axis=0)
            myct['donor'] = samplemeta.donor
            myct[pheno] = samplemeta[pheno]
            fdrs, fdr10pt, stats, ps, globalp = test_clusters(myct, cols, pheno, 'donor', Nnull=10000)
            d.uns[f'clustercc{suffix}'] = pd.DataFrame({'corr':stats, 'p':ps}, index=pd.Series(cols, name='cluster'))
            d.uns[f'clustercc{suffix}_key'] = cluster_key
            d.uns[f'clustercc{suffix}_minp'] = np.min(ps*len(ps))
            d.uns[f'clustercc{suffix}_globalp'] = globalp
            d.uns[f'clustercc{suffix}_npos'] = d.obs[cluster_key].isin(cols[stats > fdr10pt]).sum()
            d.uns[f'clustercc{suffix}_nneg'] = d.obs[cluster_key].isin(cols[stats < -fdr10pt]).sum()
            return stats, fdr10pt

    # Main phenotype
    stats, fdr10pt = store_results('', 'case')

    # Secondary phenotype (if provided)
    mask = stats > fdr10pt
    if secondary_pheno and mask.sum() > 0:
        store_results('2', secondary_pheno, mask)
    else:
        store_results('2', secondary_pheno, mask=np.zeros(len(clusts), dtype=bool))

def test_mn_cc(d, samplemeta, secondary_pheno=None):
    def store_results(suffix, pheno, mask):
        myd = d[mask].copy() if mask is not None else d
        if len(myd) > 100:
            if len(myd) < len(d):
                sc.pp.neighbors(myd)
            d.uns[f'mncc{suffix}_p'], D = vima.association([myd], samplemeta[pheno], 'sid', donorids=samplemeta.donor,
                                                key_added=f'mncoef{suffix}', make_umap=False, allow_low_sample_size=True)
            d.obs[f'mncoef{suffix}_fdr'] = D.obs.mncoef_fdr
            d.obs[f'mncoef{suffix}'] = D.obs.mncoef
            d.uns[f'mncc{suffix}_npos'] = ((d.obs.mncoef_fdr <= 0.1) & (d.obs.mncoef > 0)).sum()
            d.uns[f'mncc{suffix}_nneg'] = ((d.obs.mncoef_fdr <= 0.1) & (d.obs.mncoef < 0)).sum()
    
    store_results('', 'case', None)

    mask = (d.obs.mncoef_fdr <= 0.1) & (d.obs.mncoef > 0)
    if secondary_pheno and mask.sum() > 0:
        print(mask.sum(), 'positive correlations for primary phenotype')
        store_results('2', secondary_pheno, mask)

In [30]:
def assess(dsetname, samplemeta, secondary_pheno=None):
    embeddings = glob.glob(f'_embeddings/{dsetname}*.h5ad')
    for embedding in embeddings:
        fname = os.path.basename(embedding)
        method = fname.split('_')[1]
        harm = fname.split('_')[2]
        print(method, harm)
        d = sc.read_h5ad(embedding)
        integration(d)
        test_cluster_cc(d, samplemeta, secondary_pheno=secondary_pheno)
        test_mn_cc(d, samplemeta, secondary_pheno=secondary_pheno)
        print(method, harm)
        print(f'\tmed perp: {d.obs.perplexity.median()}')
        print(f'\tcluster minp: {d.uns['clustercc_minp']}, cluster globalp: {d.uns['clustercc_globalp']}, npos: {d.uns['clustercc_npos']} nneg: {d.uns['clustercc_nneg']}')
        print(f'\tMN p: {d.uns['mncc_p']}, npos: {d.uns['mncc_npos']} nneg: {d.uns['mncc_nneg']}')
        if 'clustercc2_minp' in d.uns:
            print(f'\tcluster2 minp: {d.uns['clustercc2_minp']}, cluster2 globalp: {d.uns['clustercc2_globalp']}')
        if 'mncc2_p' in d.uns:
            print(f'\tMN2 p: {d.uns['mncc2_p']}, npos: {d.uns['mncc2_npos']} nneg: {d.uns['mncc2_nneg']}')
        print('======')
        d.write(f'_results/{fname}')

# ALZ

In [34]:
# generate samplemeta with one row per sample (rather than per donor)
cells = pd.read_csv('../../ALZ/alz-data/SEAAD_MTG_MERFISH_metadata.2024-05-03.noblanks.harmonized.txt',
                         sep='\t')
cells['donor'] = cells.index.str.split('_').str[0]
cells['sid'] = cells.index.str.split('_').str[1]
sid_to_donor = cells[['sid', 'donor']].drop_duplicates()

samplemeta = pd.read_csv('../../ALZ/alz-data/sea-ad_cohort_donor_metadata_encoded_20240924.tsv',
                         sep='\t').drop(columns=['Donor ID']).set_index('donor', drop=True)
samplemeta = pd.merge(sid_to_donor, samplemeta, left_on='donor', right_index=True, how='left').set_index('sid', drop=True)
samplemeta['case'] = samplemeta['Consensus Clinical Dx (choice=Control)'] != 'Checked'

In [35]:
assess('ALZ', samplemeta)

patchavgmm noharm.h5ad
Using leiden1 for clustering. There are 29 clusters


/Users/yakir/Dropbox/py/cna/src/cna/tools/_stats.py:26: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Yg = np.array([Y[G==g][0] for g in Gu])


computing MAMs
performing association test
P = 0.49875012498750126
patchavgmm noharm.h5ad
	med perp: 0.06378892064094543
	cluster minp: 0.6263373662633737, cluster globalp: 0.41555844415558446, npos: 0 nneg: 0
	MN p: 0.49875012498750126, npos: 0 nneg: 0
stagate noharm.h5ad
Using leiden_1 for clustering. There are 63 clusters


/Users/yakir/Dropbox/py/cna/src/cna/tools/_stats.py:26: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Yg = np.array([Y[G==g][0] for g in Gu])


computing MAMs
There are 200115 microniches, which is greater than the maximum allowed (200000). Downsampling to 200000 microniches.
performing association test
P = 0.3902609739026097
stagate noharm.h5ad
	med perp: 0.01969745196402073
	cluster minp: 0.5354464553544646, cluster globalp: 0.15628437156284372, npos: 0 nneg: 3835
	MN p: 0.3902609739026097, npos: 0 nneg: 0
patchcelltypeabundance noharm.h5ad
Using leiden1 for clustering. There are 47 clusters


...

computing MAMs
There are 1652992 microniches, which is greater than the maximum allowed (200000). Downsampling to 200000 microniches.
performing association test
P = 0.35326467353264673
utag noharm.h5ad
	med perp: 0.13040640950202942
	cluster minp: 0.1817818218178182, cluster globalp: 0.24977502249775022, npos: 0 nneg: 0
	MN p: 0.35326467353264673, npos: 0 nneg: 167
cellcharter noharm.h5ad
Using cluster_method for clustering. There are 4 clusters


/Users/yakir/Dropbox/py/cna/src/cna/tools/_stats.py:26: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Yg = np.array([Y[G==g][0] for g in Gu])


computing MAMs
There are 1652992 microniches, which is greater than the maximum allowed (200000). Downsampling to 200000 microniches.
performing association test
P = 0.386961303869613
cellcharter noharm.h5ad
	med perp: 0.1030743345618248
	cluster minp: 0.6307369263073692, cluster globalp: 0.19308069193080693, npos: 0 nneg: 0
	MN p: 0.386961303869613, npos: 0 nneg: 0
canvas noharm.h5ad
Using leiden_1 for clustering. There are 23 clusters


/Users/yakir/Dropbox/py/cna/src/cna/tools/_stats.py:26: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Yg = np.array([Y[G==g][0] for g in Gu])


computing MAMs
performing association test
P = 0.24847515248475152
canvas noharm.h5ad
	med perp: 0.1584242284297943
	cluster minp: 0.8095190480951906, cluster globalp: 0.22717728227177283, npos: 0 nneg: 0
	MN p: 0.24847515248475152, npos: 0 nneg: 0


# RA

In [32]:
# read in and reformat sample metadata
fullmeta = pd.read_csv('../../RA/BHAM-data/ihc-metadata.csv').set_index('subject_id')[['CTAP']]
fullmeta.index = fullmeta.index.str.replace('V0', '') # reformat sample names
fullmeta['fstar'] = (fullmeta.CTAP == 'F') | (fullmeta.CTAP == 'T + F') | (fullmeta.CTAP == 'E + F + M') # define our phenotype

# change samplemeta so that each row is a sample rather than a donor
inourdata = sc.read_h5ad('_embeddings/RA_stagate_noharm.h5ad').obs[['sid','donor']].drop_duplicates()
samplemeta = pd.merge(inourdata, fullmeta, left_on='donor', right_index=True, how='left').set_index('sid', drop=True)
samplemeta['case'] = samplemeta.fstar

In [33]:
assess('RA', samplemeta)

canvas noharm.h5ad
Using leiden_1 for clustering. There are 36 clusters


/Users/yakir/Dropbox/py/cna/src/cna/tools/_stats.py:26: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Yg = np.array([Y[G==g][0] for g in Gu])


computing MAMs
performing association test
P = 0.1464853514648535
canvas noharm.h5ad
	med perp: 0.16841016709804535
	cluster minp: 0.10078992100789921, cluster globalp: 0.1653834616538346, npos: 0 nneg: 0
	MN p: 0.1464853514648535, npos: 0 nneg: 0
patchavgmm noharm.h5ad
Using leiden1 for clustering. There are 37 clusters


/Users/yakir/Dropbox/py/cna/src/cna/tools/_stats.py:26: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Yg = np.array([Y[G==g][0] for g in Gu])


computing MAMs
performing association test
P = 0.0023997600239976003
patchavgmm noharm.h5ad
	med perp: 0.1218021810054779
	cluster minp: 0.603039696030397, cluster globalp: 0.035896410358964105, npos: 0 nneg: 0
	MN p: 0.0023997600239976003, npos: 0 nneg: 0
stagate noharm.h5ad
Using leiden_1 for clustering. There are 74 clusters


/Users/yakir/Dropbox/py/cna/src/cna/tools/_stats.py:26: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Yg = np.array([Y[G==g][0] for g in Gu])


computing MAMs
There are 366976 microniches, which is greater than the maximum allowed (200000). Downsampling to 200000 microniches.
performing association test
P = 0.0192980701929807
stagate noharm.h5ad
	med perp: 0.06294918060302734
	cluster minp: 1.3096690330966905, cluster globalp: 0.017298270172982702, npos: 16047 nneg: 18250
	MN p: 0.0192980701929807, npos: 0 nneg: 0


# UC

In [29]:
# read in sample metadata
samplemeta = pd.read_csv('../../UC/UC-data/2024_10_16_UC_Patient_Metadata.csv').rename(columns={'NEW Label':'sid', 'Patient.ID':'donor'}).set_index('sid', drop=True)
samplemeta.donor = samplemeta.donor.astype('str')
samplemeta['case'] = (samplemeta.Status == 'UC').astype('float')
samplemeta.loc[(samplemeta.TNFnow == 'y') | (samplemeta.TNFprior == 'y'), 'case'] = np.nan
samplemeta['TNF'] = (samplemeta.TNFprior == 'y').astype('float')
samplemeta.loc[samplemeta.case == 0, 'TNF'] = np.nan

In [31]:
assess('UC', samplemeta, secondary_pheno='TNF')

stagate noharm.h5ad
Using leiden_1 for clustering. There are 62 clusters


/var/folders/w_/x2_v44_93nq_b3svp199t1rw0000gn/T/ipykernel_7847/797230936.py:11: RuntimeWarning: invalid value encountered in divide
  X = (X - X.mean(axis=0))/X.std(axis=0)
/Users/yakir/Dropbox/py/cna/src/cna/tools/_stats.py:26: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Yg = np.array([Y[G==g][0] for g in Gu])


computing MAMs
performing association test
P = 0.35026497350264973
stagate noharm.h5ad
	med perp: 0.03310178965330124
	cluster minp: 0.1673832616738326, cluster globalp: 0.0504949505049495, npos: 0 nneg: 4579
	MN p: 0.35026497350264973, npos: 0 nneg: 0
patchavgmm noharm.h5ad
Using leiden1 for clustering. There are 32 clusters


/Users/yakir/Dropbox/py/cna/src/cna/tools/_stats.py:26: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Yg = np.array([Y[G==g][0] for g in Gu])


computing MAMs
performing association test
P = 0.0030996900309969004
1 positive correlations for primary phenotype
patchavgmm noharm.h5ad
	med perp: 0.06082025542855263
	cluster minp: 0.015998400159984, cluster globalp: 0.0026997300269973002, npos: 0 nneg: 1551
	MN p: 0.0030996900309969004, npos: 1 nneg: 268
patchcelltypeabundance noharm.h5ad
Using leiden1 for clustering. There are 35 clusters


...

computing MAMs
performing association test
P = 0.13708629137086292
patchcelltypeabundance noharm.h5ad
	med perp: 0.06179821491241455
	cluster minp: 0.43045695430456954, cluster globalp: 0.13258674132586742, npos: 0 nneg: 0
	MN p: 0.13708629137086292, npos: 0 nneg: 0
canvas noharm.h5ad
Using leiden_1 for clustering. There are 19 clusters


/var/folders/w_/x2_v44_93nq_b3svp199t1rw0000gn/T/ipykernel_7847/797230936.py:11: RuntimeWarning: invalid value encountered in divide
  X = (X - X.mean(axis=0))/X.std(axis=0)
/Users/yakir/Dropbox/py/cna/src/cna/tools/_stats.py:26: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Yg = np.array([Y[G==g][0] for g in Gu])


computing MAMs
performing association test
P = 0.1421857814218578
canvas noharm.h5ad
	med perp: 0.05356326326727867
	cluster minp: 0.47495250474952505, cluster globalp: 0.09499050094990501, npos: 0 nneg: 0
	MN p: 0.1421857814218578, npos: 0 nneg: 0
utag noharm.h5ad
Using leiden_0.3 for clustering. There are 41 clusters


/var/folders/w_/x2_v44_93nq_b3svp199t1rw0000gn/T/ipykernel_7847/797230936.py:11: RuntimeWarning: invalid value encountered in divide
  X = (X - X.mean(axis=0))/X.std(axis=0)
/Users/yakir/Dropbox/py/cna/src/cna/tools/_stats.py:26: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Yg = np.array([Y[G==g][0] for g in Gu])


computing MAMs
There are 1522054 microniches, which is greater than the maximum allowed (200000). Downsampling to 200000 microniches.
performing association test
P = 0.07089291070892911
utag noharm.h5ad
	med perp: 0.051317933946847916
	cluster minp: 0.024597540245975404, cluster globalp: 0.011198880111988802, npos: 0 nneg: 413071
	MN p: 0.07089291070892911, npos: 0 nneg: 5677


# See results

# RA

In [19]:
results = []
perplexities = {}
for f in glob.glob('_results/RA*.h5ad'):
    method = os.path.basename(f).split('_')[1]
    d = sc.read_h5ad(f)
    perplexities[method] = d.obs.perplexity.values
    results.append({'method': method,
                     'p': d.uns['clustercc_globalp'],
                     'npos': d.uns['mncc_npos'],
                     'nneg': d.uns['mncc_nneg']})
d = sc.read_h5ad('../RA/_results/cc_fstar.h5ad')
results.append({'method': 'vima',
                'p': d.uns['vima_p'],
                'npos': (d.obs.sig_mncoef > 0).sum(),
                'nneg': (d.obs.sig_mncoef < 0).sum()
})
results = pd.DataFrame(results).set_index('method')

In [20]:
results

,p,npos,nneg
method,,,
canvas,0.154485,0,0
patchavgmm,0.035896,0,0
stagate,0.017298,0,0
vima,0.000200,2089,3133


# UC

In [ ]:
results = []
perplexities = {}
for f in glob.glob('_results/UC*.h5ad'):
    method = os.path.basename(f).split('_')[1]
    d = sc.read_h5ad(f)
    perplexities[method] = d.obs.perplexity.values
    results.append({'method': method,
                     'p': d.uns['clustercc_globalp'],
                     'npos': d.uns['mncc_npos'],
                     'nneg': d.uns['mncc_nneg']})
d = sc.read_h5ad('../UC/_results/cc_uc.h5ad')
results.append({'method': 'vima',
                'p': d.uns['vima_p'],
                'npos': (d.obs.sig_mncoef > 0).sum(),
                'nneg': (d.obs.sig_mncoef < 0).sum()
})
results = pd.DataFrame(results).set_index('method')

In [17]:
results

,p,npos,nneg
method,,,
stagate,0.049595,0,0
patchavgmm,0.002700,1,268
patchcelltypeabundance,0.132587,0,0
canvas,0.094991,0,0
utag,0.011199,0,5677
vima,0.001700,4146,7693


# ALZ

In [36]:
results = []
perplexities = {}
for f in glob.glob('_results/ALZ*.h5ad'):
    method = os.path.basename(f).split('_')[1]
    d = sc.read_h5ad(f)
    perplexities[method] = d.obs.perplexity.values
    results.append({'method': method,
                     'p': d.uns['clustercc_globalp'],
                     'npos': d.uns['mncc_npos'],
                     'nneg': d.uns['mncc_nneg']})
d = sc.read_h5ad('../ALZ/_results/cc_dementia.h5ad')
results.append({'method': 'vima',
                'p': d.uns['vima_p'],
                'npos': (d.obs.sig_mncoef > 0).sum(),
                'nneg': (d.obs.sig_mncoef < 0).sum()
})
results = pd.DataFrame(results).set_index('method')

In [37]:
results

,p,npos,nneg
method,,,
patchavgmm,0.415558,0,0
stagate,0.156284,0,0
patchcelltypeabundance,0.543446,0,0
utag,0.249775,0,167
cellcharter,0.193081,0,0
canvas,0.227177,0,0
vima,0.005399,22953,17204
